In [1]:
'''
LPM embeddings
'''

'\nLPM embeddings\n'

In [2]:
import anndata as ad
import pandas as pd
import numpy as np

In [3]:
import perturb_lib as plib

In [4]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [40]:
de_train = ad.read_h5ad('./data/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')

In [41]:
compounds = pd.read_csv('./data/embeddings/resources/compounds.csv')

In [42]:
embeddings = np.load('./data/embeddings/resources/embeddings.npy')

In [43]:
compounds['LPM_emb'] = embeddings.tolist()

In [44]:
compounds = compounds.merge(de_train.obs.drop_duplicates('sm_name')[['sm_name', 'SMILES']].reset_index(drop=True), on='sm_name', how='left').rename(columns={'SMILES': 'smiles', 'sm_name': 'perturbagen'}).drop(columns=['index'])

In [45]:
compounds['ECFP:2'] = smiles_to_fingerprints(compounds['smiles'])

In [47]:
compounds.to_pickle("./data/benchmark/resources/datasets/neurips-2023-data/op3_emb.pkl")